# Training with a Built-in Algorithm (XGBoost)

No training code to write or maintain — SageMaker owns the whole training loop. 

Tradeoff: you're limited to what the built-in algorithm supports.

In [ ]:
import boto3
import sagemaker
import pandas as pd


## TODO: Update the following variables with your own values
REGION = ""
BUCKET_NAME = ""
EXECUTION_ROLE_ARN = ""


boto_session = boto3.Session(region_name=REGION)
sm_session = sagemaker.Session(boto_session=boto_session)

In [ ]:
# NOTE: Provide data path for loading
# 1. Provide path to the raw data file in S3. The path should be in the format: s3://<bucket_name>/<path_to_file>
# 2. Provide path to the raw data file in local file system. The path should be in the format: <path_to_file>
# raw_s3_uri = f"s3://{BUCKET_NAME}/raw/PS_20174392719_1491204439457_log.csv"

raw_data_path = "" #path_to_local_data_file
df = pd.read_csv(raw_data_path)

## Feature Engineering

Steps: drop the balance columns (leaky per the EDA caveat), move `isFraud` to column 0 —
**required** since the built-in XGBoost container has no concept of column names, it always
treats column 0 as the target — then split, write headerless CSVs, and upload to S3.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
cols_to_drop = ["oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest"]
df_new = df.drop(columns=cols_to_drop)

# IMPORTANT: Move the target column to the first column in the dataframe. This is a requirement for the SageMaker built-in XGBoost algorithm.
feature_cols = [c for c in df_new.columns if c != "isFraud"]
df_new = df_new[["isFraud"] + feature_cols]

train, val = train_test_split(
    df_new, test_size=0.2, stratify=df_new["isFraud"], random_state=42
)
train.to_csv("train.csv", header=False, index=False)
val.to_csv("validation.csv", header=False, index=False)

train_s3 = sm_session.upload_data(
    "train.csv", bucket=BUCKET_NAME, key_prefix="processed/train"
)
val_s3 = sm_session.upload_data(
    "validation.csv", bucket=BUCKET_NAME, key_prefix="processed/validation"
)

noleak_scale_pos_weight = ((train["isFraud"] == 0).sum() / (train["isFraud"] == 1).sum())

## Quick sanity checks

Not required for training — just verify the uploaded S3 paths and the resulting train/validation split before configuring the estimator.

In [ ]:
train_s3, val_s3

In [ ]:
train.head()

## Retrieve the built-in XGBoost container

`image_uris.retrieve` resolves the managed container URI for the given algorithm, version,
and region — no need to hardcode it.

In [ ]:
from sagemaker.estimator import Estimator
from sagemaker import image_uris
from sagemaker.inputs import TrainingInput

xgboost_image_uri = image_uris.retrieve("xgboost", REGION, version="1.7-1")
print(xgboost_image_uri)

In [ ]:
train['isFraud'].value_counts()

In [ ]:
train.head()

## Configure and launch the training job

1. `use_spot_instances` opts into Spot pricing for the training instance (cheaper, at the risk
of interruption — `max_wait` bounds how long SageMaker will wait for capacity).<br> 
2. `eval_metric="auc"` matters more here. Since the dataset is highly imbalanced, accuracy would be misleading <br>
3. `scale_pos_weight` (legit-count / fraud-count in the training set) rebalances the loss so the model doesn't just learn to always predict "legit".

In [ ]:
estimator = Estimator(
    image_uri=xgboost_image_uri,
    role=EXECUTION_ROLE_ARN,
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=f"s3://{BUCKET_NAME}/models/builtin-xgboost/",
    sagemaker_session=sm_session,
    use_spot_instances=True,
    max_run=3600,
    max_wait=7200,
)
estimator.set_hyperparameters(
    objective="binary:logistic",
    eval_metric="auc",
    num_round=100,
    max_depth=5,
    eta=0.2,
    scale_pos_weight=noleak_scale_pos_weight,
)

estimator.fit({
    "train": TrainingInput(train_s3, content_type="text/csv"),
    "validation": TrainingInput(val_s3, content_type="text/csv"),
})